In [1]:
!pip -q install unidecode emoji==2.8.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.9/358.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 10.6 MB/s eta 0:00:00


In [4]:
# ALL packages (transitive deps too) – big but exact
!pip freeze > requirements.txt


In [3]:
# Create a pinned requirements.txt from the current Colab session
import importlib, sys, pathlib

pkgs = [
    "transformers", "torch", "tokenizers",
    "numpy", "pandas", "matplotlib", "unidecode emoji",
]

lines = []
for p in pkgs:
    try:
        m = importlib.import_module(p)
        ver = getattr(m, "__version__", None)
        if ver is None and p == "matplotlib":
            import matplotlib
            ver = matplotlib.__version__
        assert ver, f"Couldn't get version for {p}"
        lines.append(f"{p}=={ver}")
    except Exception as e:
        print(f"[warn] {p}: {e}")

pathlib.Path("requirements.txt").write_text("\n".join(lines) + "\n")
print("Wrote requirements.txt:\n" + "\n".join(lines))


[warn] unidecode emoji: No module named 'unidecode emoji'
Wrote requirements.txt:
transformers==4.56.1
torch==2.8.0+cu126
tokenizers==0.22.0
numpy==2.0.2
pandas==2.2.2
matplotlib==3.10.0


In [2]:
# Q2. Robustness to Messy Inputs

# 1) Builds a 50-item gold QA set (US state -> capital).
# 2) Adds 4 noise types at 2 strengths each.
# 3) Evaluates FLAN-T5-small (HF Transformers) with EM and token-F1.
# 4) Reports deltas vs clean (absolute points and %), with Wilson 95% CIs.
# 5) Tests a robustness intervention (preprocessor + prompt template) on a 20-item subset.
# 6) Saves: results.csv, results_subset.csv, heatmaps,t.

# Setup & Dependencies


import math, random, re, unicodedata, string, os, json, textwrap
from dataclasses import dataclass
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from unidecode import unidecode
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

os.makedirs("outputs", exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Data: 50 Clean Prompts

# Gold answer list for US capitals
STATE_TO_CAPITAL = {
    "Alabama": "Montgomery",
    "Alaska": "Juneau",
    "Arizona": "Phoenix",
    "Arkansas": "Little Rock",
    "California": "Sacramento",
    "Colorado": "Denver",
    "Connecticut": "Hartford",
    "Delaware": "Dover",
    "Florida": "Tallahassee",
    "Georgia": "Atlanta",
    "Hawaii": "Honolulu",
    "Idaho": "Boise",
    "Illinois": "Springfield",
    "Indiana": "Indianapolis",
    "Iowa": "Des Moines",
    "Kansas": "Topeka",
    "Kentucky": "Frankfort",
    "Louisiana": "Baton Rouge",
    "Maine": "Augusta",
    "Maryland": "Annapolis",
    "Massachusetts": "Boston",
    "Michigan": "Lansing",
    "Minnesota": "Saint Paul",
    "Mississippi": "Jackson",
    "Missouri": "Jefferson City",
    "Montana": "Helena",
    "Nebraska": "Lincoln",
    "Nevada": "Carson City",
    "New Hampshire": "Concord",
    "New Jersey": "Trenton",
    "New Mexico": "Santa Fe",
    "New York": "Albany",
    "North Carolina": "Raleigh",
    "North Dakota": "Bismarck",
    "Ohio": "Columbus",
    "Oklahoma": "Oklahoma City",
    "Oregon": "Salem",
    "Pennsylvania": "Harrisburg",
    "Rhode Island": "Providence",
    "South Carolina": "Columbia",
    "South Dakota": "Pierre",
    "Tennessee": "Nashville",
    "Texas": "Austin",
    "Utah": "Salt Lake City",
    "Vermont": "Montpelier",
    "Virginia": "Richmond",
    "Washington": "Olympia",
    "West Virginia": "Charleston",
    "Wisconsin": "Madison",
    "Wyoming": "Cheyenne",
}
STATES = list(STATE_TO_CAPITAL.keys())
CAPITALS = set(STATE_TO_CAPITAL.values())


In [3]:
def build_prompt(state: str, robust: bool=False) -> str:
    if robust:
        # Robust prompting template: warn about noise; require terse output
        return (
            "Instruction: You may see typos, extra spaces, unusual Unicode (full-width or look-alike characters), "
            "or emoji. Mentally normalize and infer the intended meaning. "
            "Respond with only the U.S. state capital, no extra words or punctuation.\n"
            f"Question: What is the capital of the U.S. state of {state}? Answer:"
        )
    else:
        return f"Question: What is the capital of the U.S. state of {state}? Answer with only the city name."

CLEAN = [{"id": i+1, "state": s, "prompt": build_prompt(s, robust=False), "gold": STATE_TO_CAPITAL[s]}
         for i, s in enumerate(STATES)]

# Noise Generators (4 types, 2 strengths)

# Utility: collapse whitespace
def collapse_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

# 1) Typos: keyboard-adjacent substitutions/insertions (QWERTY-based)
QWERTY_ROWS = ["`1234567890-=", "qwertyuiop[]\\", "asdfghjkl;'", "zxcvbnm,./"]
LOC = {}
for r, row in enumerate(QWERTY_ROWS):
    for c, ch in enumerate(row):
        LOC[ch] = (r, c)

def kb_neighbors(ch: str) -> List[str]:
    base = ch.lower()
    if base not in LOC:
        return []
    r, c = LOC[base]
    cand = []
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue
            rr, cc = r+dr, c+dc
            if 0 <= rr < len(QWERTY_ROWS) and 0 <= cc < len(QWERTY_ROWS[rr]):
                cand.append(QWERTY_ROWS[rr][cc])
    # preserve case
    out = []
    for z in cand:
        out.append(z.upper() if ch.isupper() else z)
    return out


def add_typos(text: str, level: int) -> str:
    # level 1: ~1-2 edits; level 2: ~3-5 edits
    n_letters = sum(ch.isalpha() for ch in text)
    edits = 2 if level == 1 else 5
    edits = min(edits, max(1, n_letters // 10)) if level == 1 else min(edits, max(2, n_letters // 7))
    s = list(text)
    for _ in range(edits):
        idxs = [i for i, ch in enumerate(s) if ch.isalpha()]
        if not idxs: break
        i = random.choice(idxs)
        ch = s[i]
        neigh = kb_neighbors(ch)
        if not neigh:
            continue
        if random.random() < 0.5:
            # substitution
            s[i] = random.choice(neigh)
        else:
            # insertion (neighbor)
            s.insert(i, random.choice(neigh))
    return "".join(s)


# 2) Spacing/Punctuation: inject extra spaces; drop punctuation
PUNCT = set(list(",.;:!?"))
def spacing_punct_noise(text: str, level: int) -> str:
    s = text
    # extra spaces
    n_spaces = 2 if level == 1 else 6
    for _ in range(n_spaces):
        # insert "  " after a random word boundary
        m = list(re.finditer(r"\s", s))
        if not m:
            s = s + "  "
        else:
            pos = random.choice(m).start()
            s = s[:pos] + "  " + s[pos:]
    # drop punctuation
    if level == 1:
        # drop 1-2 punctuation marks if present
        for _ in range(2):
            positions = [i for i, ch in enumerate(s) if ch in PUNCT]
            if positions:
                j = random.choice(positions)
                s = s[:j] + s[j+1:]
    else:
        s = "".join(ch for ch in s if ch not in PUNCT)
    # sometimes trim/expand leading/trailing spaces
    if random.random() < 0.5:
        s = " " + s
    if random.random() < 0.5:
        s = s + "   "
    return s


# 3) Unicode Confusables & width mixing
CONFUSABLES = {
    # lowercase
    "a":"α", "e":"е", "i":"і", "o":"ο", "p":"р", "c":"с", "x":"х", "y":"у",
    "h":"һ", "k":"κ", "m":"ｍ", "n":"ｎ", "s":"ѕ", "t":"ｔ",
    # uppercase
    "A":"Α", "B":"Β", "C":"С", "E":"Е", "H":"Η", "I":"Ι", "K":"Κ", "M":"Μ", "O":"Ο", "P":"Ρ", "T":"Τ", "X":"Χ", "Y":"Υ",
    # digits/letters
    "0":"O", "O":"0", "1":"l", "l":"1", "5":"S", "S":"5"
}

def ascii_to_fullwidth(ch: str) -> str:
    # converts ASCII visible range to full-width
    code = ord(ch)
    if 33 <= code <= 126:
        return chr(code + 0xFEE0)
    return ch

def unicode_confusables_noise(text: str, level: int) -> str:
    # level controls replacement probability
    p = 0.15 if level == 1 else 0.35
    out = []
    for ch in text:
        if ch in CONFUSABLES and random.random() < p:
            out.append(CONFUSABLES[ch])
        elif ch.isascii() and ch.isalnum() and random.random() < (p/3.0):
            out.append(ascii_to_fullwidth(ch))
        else:
            out.append(ch)
    return "".join(out)


# 4) Emoji/Icons: sprinkle neutral emoji
EMOJI_POOL = ["🙂","✨","🔹","📍","⭐","✅","🧭","📎","🗺️","⚪"]
EMOJI_RE = re.compile("|".join(map(re.escape, EMOJI_POOL)))
def emoji_noise(text: str, level: int) -> str:
    s = text
    k = 2 if level == 1 else 6
    for _ in range(k):
        pos = random.randrange(0, len(s)+1)
        s = s[:pos] + " " + random.choice(EMOJI_POOL) + " " + s[pos:]
    # Try to bracket the state name if present
    m = re.search(r"state of ([A-Za-z ]+)\?", s)
    if m:
        name = m.group(1)
        s = s.replace(name, f"🙂 {name} 📍") if level == 1 else s.replace(name, f"✨🙂 {name} 📍⭐")
    return collapse_ws(s)

# Dispatcher
NOISE_TYPES = {
    "typos": add_typos,
    "spacing_punct": spacing_punct_noise,
    "unicode_confusables": unicode_confusables_noise,
    "emoji": emoji_noise
}
NOISE_LEVELS = [1, 2]  # 1 = light, 2 = heavy

# Normalization & Metrics (EM, token-F1)

def normalize_for_eval(s: str) -> str:
    # 1) NFKC normalize; 2) strip diacritics; 3) lower; 4) remove emoji & punctuation; 5) collapse whitespace
    s_nfkc = unicodedata.normalize("NFKC", s)
    s_ascii = unidecode(s_nfkc)
    s_noemoji = EMOJI_RE.sub(" ", s_ascii)
    s_lower = s_noemoji.lower()
    s_nopunct = "".join(ch for ch in s_lower if ch.isalnum() or ch.isspace())

    tokens = s_nopunct.split()
    tokens = ["saint" if t in {"st", "st."} else t for t in tokens]
    return collapse_ws(" ".join(tokens))

def exact_match(pred: str, gold: str) -> int:
    return int(normalize_for_eval(pred) == normalize_for_eval(gold))

def token_f1(pred: str, gold: str) -> float:
    p = normalize_for_eval(pred).split()
    g = normalize_for_eval(gold).split()
    if not p or not g:
        return 0.0
    p_set, g_set = set(p), set(g)
    inter = len(p_set & g_set)
    if inter == 0:
        return 0.0
    prec = inter / len(p_set)
    rec = inter / len(g_set)
    return 2 * prec * rec / (prec + rec)

# Wilson CI (95%) for accuracy
def wilson_ci(k: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    if n == 0:
        return (0.0, 0.0)
    phat = k / n
    denom = 1 + z**2 / n
    center = (phat + z**2/(2*n)) / denom
    margin = (z * math.sqrt((phat*(1-phat) + z**2/(4*n))/n)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))



In [ ]:
# Model: FLAN-T5-small (HF Transformers)

DEVICE = 0 if torch.cuda.is_available() else -1
GEN = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    tokenizer="google/flan-t5-large",
    device=DEVICE
)

def model_predict(prompts: List[str], max_new_tokens: int = 16, batch_size: int = 8) -> List[str]:
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        gen = GEN(batch, max_new_tokens=max_new_tokens, do_sample=False)
        outs = [g["generated_text"] for g in gen]

        outs = [o.strip().split("\n")[0].strip() for o in outs]
        outputs.extend(outs)
    return outputs

# Building noisy evaluation sets

def make_noisy_rows(clean_rows: List[Dict]) -> List[Dict]:
    rows = []
    # Clean baseline
    for r in clean_rows:
        rows.append({
            "id": r["id"],
            "state": r["state"],
            "noise_type": "clean",
            "noise_level": "none",
            "prompt_in": r["prompt"],
            "gold": r["gold"]
        })
    # Noisy variants
    for r in clean_rows:
        for nt_name, fn in NOISE_TYPES.items():
            for lvl in NOISE_LEVELS:
                rows.append({
                    "id": r["id"],
                    "state": r["state"],
                    "noise_type": nt_name,
                    "noise_level": "light" if lvl == 1 else "heavy",
                    "prompt_in": fn(r["prompt"], level=lvl),
                    "gold": r["gold"]
                })
    return rows

EVAL_ROWS = make_noisy_rows(CLEAN)

#  evaluation

def evaluate(rows: List[Dict], use_robust_prompt: bool=False, preprocessor=None, subset_ids: List[int]=None,
             tag: str="full") -> pd.DataFrame:
    # Optionally restrict to subset
    data = [r for r in rows if (subset_ids is None or r["id"] in subset_ids)]
    prompts = []
    meta = []
    for r in data:
        p = r["prompt_in"]
        # If robust prompt is requested, rebuild the clean prompt with template and then re-inject EXACT SAME NOISE
        if use_robust_prompt:
            base = build_prompt(r["state"], robust=True)
            # Try to align with existing noise by reapplying the noise function if any
            if r["noise_type"] == "clean":
                noisy = base
            else:
                noisy = NOISE_TYPES[r["noise_type"]](base, 1 if r["noise_level"]=="light" else 2)
            p = noisy
        if preprocessor is not None:
            p = preprocessor(p)
        prompts.append(p)
        meta.append(r)

    preds = model_predict(prompts, max_new_tokens=16, batch_size=8)

    recs = []
    for m, pred, prompt_in in zip(meta, preds, prompts):
        em = exact_match(pred, m["gold"])
        f1 = token_f1(pred, m["gold"])
        recs.append({
            "id": m["id"],
            "noise_type": m["noise_type"],
            "noise_level": m["noise_level"],
            "prompt_in": prompt_in,
            "gold": m["gold"],
            "pred": pred,
            "correct": int(em),
            "f1": float(f1),
            "tag": tag
        })
    df = pd.DataFrame(recs)
    return df


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [5]:

# Light preprocessor

EMOJI_CLASS = re.compile(
    "["                             # broad emoji-ish range
    "\U0001F300-\U0001FAFF"         # symbols & pictographs
    "\U00002700-\U000027BF"         # dingbats
    "\U00002600-\U000026FF"         # misc symbols
    "]+", flags=re.UNICODE
)

def light_preprocess(text: str) -> str:

    s = unicodedata.normalize("NFKC", text)
    s = EMOJI_CLASS.sub(" ", s)
    s = unidecode(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# Full evaluation (50 items x 1 clean + 4 noises x 2 levels)

df_full = evaluate(EVAL_ROWS, use_robust_prompt=False, preprocessor=None, subset_ids=None, tag="baseline")
df_full.to_csv("outputs/results.csv", index=False)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [7]:

# Metrics & Heatmap (full)

def summarize(df: pd.DataFrame, title: str, save_path_png: str):
    # Accuracy by noise type/level
    grp = df.groupby(["noise_type","noise_level"])["correct"].agg(["mean","count","sum"]).reset_index()
    # Clean baseline
    base = grp[(grp["noise_type"]=="clean") & (grp["noise_level"]=="none")]["mean"].iloc[0]
    base_n = grp[(grp["noise_type"]=="clean") & (grp["noise_level"]=="none")]["count"].iloc[0]
    base_k = grp[(grp["noise_type"]=="clean") & (grp["noise_level"]=="none")]["sum"].iloc[0]
    base_lo, base_hi = wilson_ci(int(base_k), int(base_n))
    print(f"\n=== {title} ===")
    print(f"Clean baseline accuracy: {base:.3f} (95% CI {base_lo:.3f}–{base_hi:.3f}), n={base_n}")

    # Prepare a table for display
    idx = []
    rows = []
    for nt in ["clean", "typos", "spacing_punct", "unicode_confusables", "emoji"]:
        row = []
        for lvl in ["none", "light", "heavy"]:
            subset = grp[(grp["noise_type"]==nt) & (grp["noise_level"]==lvl)]
            if not len(subset):
                row.append(np.nan)
            else:
                row.append(float(subset["mean"].iloc[0]))
        idx.append(nt)
        rows.append(row)
    acc_mat = np.array(rows)
    df_tab = pd.DataFrame(acc_mat, index=idx, columns=["none","light","heavy"])
    print("\nAccuracy grid (rows=noise type, cols=strength):\n")
    print(df_tab.round(3))

    # Degradation vs clean (absolute pts and % drop)
    print("\nDegradation vs clean baseline:")
    for nt in ["typos","spacing_punct","unicode_confusables","emoji"]:
        for lvl in ["light","heavy"]:
            cur = df_tab.loc[nt, lvl]
            if not np.isnan(cur):
                abs_pts = (cur - base) * 100.0
                pct_drop = (0.0 if base==0 else (base - cur) / base * 100.0)
                print(f"{nt:20s} {lvl:5s}  Δ={abs_pts:+.1f} pts,  drop={pct_drop:.1f}%")

    # Heatmap (exclude 'clean' row to emphasize degradation grid)
    mask = np.ones_like(acc_mat[1:,:], dtype=bool)  # use for NaN handling if any
    fig = plt.figure(figsize=(6, 4))
    plt.imshow(acc_mat[1:,:], aspect="auto")  # use default colormap
    plt.xticks([0,1,2], ["none","light","heavy"])
    plt.yticks(range(4), ["typos","spacing_punct","unicode_confusables","emoji"])
    plt.title(title + "\n(accuracy)")
    # annotate
    for r in range(4):
        for c in range(3):
            val = acc_mat[1:,:][r,c]
            if not np.isnan(val):
                plt.text(c, r, f"{val*100:.1f}%", ha="center", va="center")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(save_path_png, dpi=150)
    plt.close()
    return df_tab, base

grid_full, base_full = summarize(df_full, "Full set (50) — Baseline model", "outputs/heatmap_full.png")


# Fixed 20-item subset for reproducibility
rng = random.Random(SEED)
subset_ids = [r["id"] for r in CLEAN]
rng.shuffle(subset_ids)
subset_ids = subset_ids[:20]

df_sub_baseline = evaluate(EVAL_ROWS, use_robust_prompt=False, preprocessor=None, subset_ids=subset_ids, tag="subset_baseline")
df_sub_prompt   = evaluate(EVAL_ROWS, use_robust_prompt=True,  preprocessor=None, subset_ids=subset_ids, tag="subset_prompt")
df_sub_both     = evaluate(EVAL_ROWS, use_robust_prompt=True,  preprocessor=light_preprocess, subset_ids=subset_ids, tag="subset_preproc_prompt")

df_subset = pd.concat([df_sub_baseline, df_sub_prompt, df_sub_both], ignore_index=True)
df_subset.to_csv("outputs/results_subset.csv", index=False)

# Summaries per setting
def summarize_subset(df: pd.DataFrame, title: str, save_path_png: str):
    grp = df.groupby(["tag","noise_type","noise_level"])["correct"].agg(["mean","count","sum"]).reset_index()
    tags = ["subset_baseline","subset_prompt","subset_preproc_prompt"]
    print(f"\n=== {title} ===")
    # print small grid per tag
    for tag in tags:
        sub = grp[grp["tag"]==tag]
        base_row = sub[(sub["noise_type"]=="clean") & (sub["noise_level"]=="none")]
        if len(base_row) == 0:
            continue
        base = float(base_row["mean"].iloc[0])
        base_n = int(base_row["count"].iloc[0])
        base_k = int(base_row["sum"].iloc[0])
        lo, hi = wilson_ci(base_k, base_n)
        print(f"\n[{tag}] Clean baseline accuracy: {base:.3f} (95% CI {lo:.3f}–{hi:.3f}), n={base_n}")

        mat = []
        for nt in ["typos","spacing_punct","unicode_confusables","emoji"]:
            row=[]
            for lvl in ["none","light","heavy"]:
                subset = sub[(sub["noise_type"]==nt) & (sub["noise_level"]==lvl)]
                row.append(float(subset["mean"].iloc[0]) if len(subset) else np.nan)
            mat.append(row)
        mat = np.array(mat)

        # Heatmap per tag
        fig = plt.figure(figsize=(6,4))
        plt.imshow(mat, aspect="auto")
        plt.xticks([0,1,2], ["none","light","heavy"])
        plt.yticks(range(4), ["typos","spacing_punct","unicode_confusables","emoji"])
        plt.title(title + f"\n(accuracy) — {tag}")
        for r in range(4):
            for c in range(3):
                val = mat[r,c]
                if not np.isnan(val):
                    plt.text(c, r, f"{val*100:.1f}%", ha="center", va="center")
        plt.colorbar()
        plt.tight_layout()
        base_name = os.path.splitext(save_path_png)[0]
        out_png = f"{base_name}_{tag}.png"
        plt.savefig(out_png, dpi=150)
        plt.close()
    return grp

grp_subset = summarize_subset(df_subset, "20‑item subset — Intervention comparison", "outputs/heatmap_subset.png")


def categorize_error(pred: str, gold: str) -> str:
    norm_pred = normalize_for_eval(pred)
    norm_gold = normalize_for_eval(gold)
    if pred in CAPITALS and pred != gold:
        return "other_state_capital"
    if token_f1(pred, gold) == 1.0 and exact_match(pred, gold) == 0:
        return "normalization_only"
    if ("capital" in pred.lower()) or (len(pred.split()) > 3):
        return "extra_verbiage"
    alnum_ratio = sum(ch.isalnum() for ch in pred) / max(1, len(pred))
    if len(pred.strip()) < 2 or alnum_ratio < 0.3:
        return "unparseable_or_empty"
    return "other"

errors = df_full[df_full["correct"]==0].copy()
errors["category"] = [categorize_error(p, g) for p, g in zip(errors["pred"], errors["gold"])]
tax_counts = errors.groupby("category").size().sort_values(ascending=False)
print("\n=== Error taxonomy (top categories) ===")
print(tax_counts)

# Pick one example per category
examples = []
for cat in tax_counts.index.tolist():
    ex = errors[errors["category"]==cat].head(1)
    if len(ex):
        r = ex.iloc[0].to_dict()
        examples.append({
            "category": cat,
            "noise_type": r["noise_type"],
            "noise_level": r["noise_level"],
            "prompt_snippet": r["prompt_in"][:120] + ("…" if len(r["prompt_in"])>120 else ""),
            "gold": r["gold"],
            "pred": r["pred"]
        })
df_examples = pd.DataFrame(examples)
df_examples.to_json("outputs/error_examples.json", orient="records", force_ascii=False, indent=2)
print("\nOne example per category written to outputs/error_examples.json")





=== Full set (50) — Baseline model ===
Clean baseline accuracy: 0.240 (95% CI 0.143–0.374), n=50

Accuracy grid (rows=noise type, cols=strength):

                     none  light  heavy
clean                0.24    NaN    NaN
typos                 NaN   0.18   0.04
spacing_punct         NaN   0.20   0.20
unicode_confusables   NaN   0.08   0.02
emoji                 NaN   0.16   0.16

Degradation vs clean baseline:
typos                light  Δ=-6.0 pts,  drop=25.0%
typos                heavy  Δ=-20.0 pts,  drop=83.3%
spacing_punct        light  Δ=-4.0 pts,  drop=16.7%
spacing_punct        heavy  Δ=-4.0 pts,  drop=16.7%
unicode_confusables  light  Δ=-16.0 pts,  drop=66.7%
unicode_confusables  heavy  Δ=-22.0 pts,  drop=91.7%
emoji                light  Δ=-8.0 pts,  drop=33.3%
emoji                heavy  Δ=-8.0 pts,  drop=33.3%

=== 20‑item subset — Intervention comparison ===

[subset_baseline] Clean baseline accuracy: 0.300 (95% CI 0.145–0.519), n=20

[subset_prompt] Clean baseline ac